# LeetCode 390 — Elimination Game bằng Python

Link đề: https://leetcode.com/problems/elimination-game/ · Độ khó: **Medium** · Chủ đề: Recursion, Math

Skeleton LeetCode:

```python
class Solution:
    def lastRemaining(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có dãy `1..n`, lần lượt xóa "cách một số" từ **trái sang phải**, rồi từ **phải sang trái**, luân phiên cho đến khi còn đúng một số.
2. Nắm các công cụ Python cần dùng: **cắt lát có bước** (`arr[::2]`, `arr[1::2]`, `arr[::-1]`), `//` và `%` (tính chẵn/lẻ), dịch bit `<<` / `>>`, **cấp số cộng** (đầu, bước, số phần tử) và **đệ quy**.
3. Đi từ cách **mô phỏng bằng list** (`O(n)`, đúng nhưng không chạy nổi `n = 10^9`) → **chỉ theo dõi phần tử đầu của cấp số cộng** → **công thức đệ quy nhờ đối xứng gương** → **theo dõi cả hai đầu**.
4. Hiểu mấu chốt: sau mỗi lượt, các số còn lại **vẫn là một cấp số cộng** (đầu, bước, số lượng), số lượng **giảm một nửa**, bước **nhân đôi**, nên chỉ cần `O(log n)` lượt.
5. Tránh các lỗi kinh điển: luôn dịch phần tử đầu dù đi từ phải, cập nhật số lượng sai (`(cnt+1)//2`), đệ quy sai công thức gương, mô phỏng bằng list với `n = 10^9`.

## 2. Đề bài & ví dụ

Cho danh sách `arr` gồm các số nguyên từ `1` đến `n` theo thứ tự tăng dần. Áp dụng thuật toán:

1. **Từ trái sang phải:** xóa số đầu tiên và cứ cách một số lại xóa một số, cho đến hết danh sách.
2. **Từ phải sang trái:** lặp lại, lần này xóa số **ngoài cùng bên phải** rồi cách một số xóa một số trong các số còn lại.
3. Cứ thế luân phiên trái → phải → trái ... cho đến khi còn đúng **một** số.

Cho `n`, trả về số cuối cùng còn lại.

**Ví dụ 1:** `n = 9` → `6`.
```
[1, 2, 3, 4, 5, 6, 7, 8, 9]   trái -> phải: xóa 1,3,5,7,9
[2, 4, 6, 8]                  phải -> trái: xóa 8,4
[2, 6]                        trái -> phải: xóa 2
[6]
```

**Ví dụ 2:** `n = 1` → `1`.

**Ràng buộc:** `1 <= n <= 10^9`.

Chú ý ràng buộc: `n` lên tới một tỉ, ta **không thể** tạo cả danh sách `1..n` trong bộ nhớ (cần hàng GB). Cần thuật toán gần như `O(log n)`.

In [1]:
examples = [(9, 6), (1, 1)]
for n, expected in examples:
    print(f"n={n} -> kỳ vọng {expected}")

n=9 -> kỳ vọng 6
n=1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Cắt lát có bước: `arr[a::b]`

`arr[start:stop:step]` lấy các phần tử cách nhau `step`. Với `step = 2`: `arr[::2]` lấy các chỉ số `0, 2, 4, ...` (xóa các chỉ số lẻ), còn `arr[1::2]` lấy chỉ số `1, 3, 5, ...` (xóa các chỉ số chẵn, kể cả chỉ số `0`). Với `step = -1` ta đọc ngược: `arr[::-1]` là bản đảo ngược.

Đây là cách mô phỏng "xóa cách một số": **xóa các chỉ số chẵn** = **giữ `arr[1::2]`**.

In [2]:
arr = [1, 2, 3, 4, 5, 6, 7, 8, 9]
print(arr[::2])        # [1, 3, 5, 7, 9]  (chỉ số 0,2,4,6,8)
print(arr[1::2])       # [2, 4, 6, 8]     (giữ chỉ số lẻ = xóa 1,3,5,7,9)
print(arr[::-1])       # [9, 8, 7, 6, 5, 4, 3, 2, 1]
print(arr[::-1][1::2]) # [8, 6, 4, 2]     (đọc từ phải, bỏ phần tử ngoài cùng, giữ cách một)

[1, 3, 5, 7, 9]
[2, 4, 6, 8]
[9, 8, 7, 6, 5, 4, 3, 2, 1]
[8, 6, 4, 2]


### 3.2. Chia nguyên `//`, phần dư `%` và tính chẵn/lẻ

`a // b` là phép chia lấy phần nguyên (làm tròn xuống với số dương), `a % 2` cho `0` nếu `a` chẵn, `1` nếu lẻ. Trong bài này số phần tử còn lại sau mỗi lượt là `cnt // 2`: nếu `cnt = 9` thì còn `4`, nếu `cnt = 8` cũng còn `4` (xóa `5` phần tử hoặc `4` phần tử tương ứng). Tính chẵn/lẻ của `cnt` quyết định phần tử **đầu** có bị xóa hay không khi đi từ phải.

In [3]:
for cnt in [9, 8, 7, 6, 1]:
    print(f"cnt={cnt}: còn lại {cnt // 2}, bị xóa {cnt - cnt // 2}, lẻ? {cnt % 2 == 1}")

cnt=9: còn lại 4, bị xóa 5, lẻ? True
cnt=8: còn lại 4, bị xóa 4, lẻ? False
cnt=7: còn lại 3, bị xóa 4, lẻ? True
cnt=6: còn lại 3, bị xóa 3, lẻ? False
cnt=1: còn lại 0, bị xóa 1, lẻ? True


### 3.3. Cấp số cộng: (đầu, bước, số lượng)

Dãy `2, 4, 6, 8` chính là một **cấp số cộng**: phần tử đầu `first = 2`, bước `step = 2`, số lượng `cnt = 4`. Phần tử thứ `k` (đếm từ 0) là `first + k * step`; phần tử cuối là `first + (cnt - 1) * step`. Ta không cần liệt kê cả dãy, chỉ cần ba con số!

In [4]:
first, step, cnt = 2, 2, 4
print([first + k * step for k in range(cnt)])      # [2, 4, 6, 8]
print("phần tử cuối:", first + (cnt - 1) * step)    # 8
# dãy 1..9 là first=1, step=1, cnt=9
print([1 + k * 1 for k in range(9)])

[2, 4, 6, 8]
phần tử cuối: 8
[1, 2, 3, 4, 5, 6, 7, 8, 9]


### 3.4. Dịch bit `<<` và `>>`

`x << 1` nhân đôi `x` (`x * 2`), `x >> 1` chia nguyên cho `2` (`x // 2`). Chúng nhanh và gọn khi bạn muốn "nhân đôi / chia đôi". Ta dùng `step <<= 1` và `cnt >>= 1` (viết tắt của `step = step << 1`).

In [5]:
step, cnt = 1, 9
for _ in range(3):
    step <<= 1
    cnt >>= 1
    print("step =", step, "cnt =", cnt)       # (2,4) (4,2) (8,1)

step = 2 cnt = 4
step = 4 cnt = 2
step = 8 cnt = 1


### 3.5. Đệ quy: hàm gọi lại chính nó

Hàm **đệ quy** giải bài toán lớn bằng cách gọi lại chính nó trên bài toán nhỏ hơn, và phải có **trường hợp cơ sở** (base case) để dừng. Ví dụ đếm số lượt chia đôi cho tới khi còn `1`: `halvings(1) = 0`, `halvings(n) = 1 + halvings(n // 2)`. Độ sâu đệ quy là số lần chia đôi, tức `O(log n)` — với `n = 10^9` chỉ khoảng 30 tầng, rất an toàn.

In [6]:
def halvings(n):
    if n == 1:                 # base case
        return 0
    return 1 + halvings(n // 2)


print(halvings(9), halvings(10**9))       # 3 29

3 29


### 3.6. Đối xứng gương

Nếu ta đánh số lại danh sách theo chiều **ngược** (đọc từ phải sang trái), số `x` ở vị trí `x` (tính từ trái, 1-based) trong danh sách `1..m` trở thành vị trí `m + 1 - x`. Nên *"đi từ phải sang trái trên `1..m`"* chính là *"đi từ trái sang phải trên danh sách bị gương"* — và kết quả cuối cùng cũng bị gương: `x ↔ m + 1 - x`. Ta sẽ dùng ý này ở Cách 3.

In [7]:
m = 5
mirror = [m + 1 - x for x in range(1, m + 1)]
print(mirror)                 # [5, 4, 3, 2, 1]  (gương của 1..5)
print(m + 1 - 2)              # số 2 <-> số 4

[5, 4, 3, 2, 1]
4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính tay vài trường hợp nhỏ để "cảm" quy luật. Viết `L` là lượt trái → phải, `R` là phải → trái.

**n = 9:** `[1..9]` →L→ `[2,4,6,8]` →R→ `[2,6]` →L→ `[6]`. Đáp án `6`.

**n = 7:** `[1..7]` →L→ `[2,4,6]` →R→ xóa 6, 2: `[4]`. Đáp án `4`.

**n = 12:** `[1..12]` →L→ `[2,4,6,8,10,12]` →R→ xóa 12, 8, 4: `[2,6,10]` →L→ xóa 2, 10: `[6]`. Đáp án `6`.

Nhận xét quan trọng:

1. Sau lượt **L** luôn còn `2, 4, 6, ...` (cấp số cộng đầu `2`, bước `2`).
2. Sau lượt **R**, các số còn lại cũng cách đều nhau (bước tăng gấp đôi). Số phần tử mỗi lượt giảm còn `cnt // 2`.
3. Vì số phần tử giảm một nửa mỗi lượt nên chỉ cần khoảng `log2(n)` lượt; không cần xóa từng phần tử.
4. Câu hỏi duy nhất ở mỗi lượt: **phần tử đầu tiên của cấp số mới là gì?**

In [8]:
def one_round(arr, from_left):
    """Mô phỏng một lượt xóa trên list."""
    if from_left:
        return arr[1::2]
    return arr[::-1][1::2][::-1]


arr = list(range(1, 10))
left = True
while len(arr) > 1:
    print(("trái->phải" if left else "phải->trái"), arr)
    arr = one_round(arr, left)
    left = not left
print("còn lại:", arr)

trái->phải [1, 2, 3, 4, 5, 6, 7, 8, 9]
phải->trái [2, 4, 6, 8]
trái->phải [2, 6]
còn lại: [6]


## 5. Cách 1 — Brute Force: mô phỏng bằng list

### Ý tưởng

Làm đúng như đề: tạo `arr = list(range(1, n + 1))`, lặp: nếu lượt trái → phải thì `arr = arr[1::2]`; nếu lượt phải → trái thì đảo ngược, giữ `[1::2]` rồi đảo lại. Dừng khi `len(arr) == 1`.

Kiểm tra lượt phải → trái với `[2, 4, 6, 8]`: đảo `[8, 6, 4, 2]`, `[1::2] = [6, 2]`, đảo lại `[2, 6]` ✓.

In [9]:
def last_remaining_brute(n):
    arr = list(range(1, n + 1))
    from_left = True
    while len(arr) > 1:
        if from_left:
            arr = arr[1::2]
        else:
            arr = arr[::-1][1::2][::-1]
        from_left = not from_left
    return arr[0]


for n, expected in examples:
    print(n, last_remaining_brute(n), expected)
print([last_remaining_brute(n) for n in range(1, 21)])

9 6 6
1 1 1
[1, 2, 2, 2, 2, 4, 4, 6, 6, 8, 8, 6, 6, 8, 8, 6, 6, 8, 8, 6]


### Độ phức tạp & nút thắt

- Tổng số phần tử được tạo ra: `n + n/2 + n/4 + ... ≈ 2n` ⇒ **Time `O(n)`**, **Space `O(n)`**.
- Với `n = 10^9`: một list Python `10^9` số nguyên cần hàng chục GB RAM và rất lâu — **không chạy nổi**.

**Nút thắt:** ta tạo cả danh sách trong khi mỗi lượt chỉ cần biết "phần tử đầu". Ta đã thấy danh sách luôn là cấp số cộng ⇒ chỉ cần theo dõi `(first, step, cnt)`.

## 6. Ý tưởng thuật toán / bước đột phá

Giữ ba biến mô tả danh sách hiện tại: `first` (phần tử đầu), `step` (bước), `cnt` (số phần tử). Ban đầu `first = 1, step = 1, cnt = n`. Sau mỗi lượt: `step *= 2`, `cnt //= 2`. Vấn đề duy nhất là `first` thay đổi thế nào.

**Lượt trái → phải:** luôn xóa phần tử đầu, nên `first += step` (phần tử còn lại đầu tiên là phần tử thứ hai cũ).

**Lượt phải → trái:** xóa phần tử cuối rồi cách một. Phần tử đầu có bị xóa không? Hãy đánh số từ **phải**: phần tử cuối là số 1 (bị xóa), số 3, số 5,... bị xóa. Phần tử đầu có thứ tự từ phải là `cnt`.
- Nếu `cnt` **lẻ** → `cnt` là số lẻ → phần tử đầu **bị xóa** ⇒ `first += step`.
- Nếu `cnt` **chẵn** → phần tử đầu **được giữ** ⇒ `first` không đổi.

Ví dụ `[2,4,6,8]` (cnt = 4 chẵn): xóa 8, 4; giữ `2, 6`, đầu vẫn là `2` ✓. Ví dụ `[2,4,6]` (cnt = 3 lẻ): xóa 6, 2; còn `[4]`, đầu dịch `+step` ✓.

Cuối cùng khi `cnt == 1`, `first` chính là đáp án.

In [10]:
# Kiểm chứng quy tắc "đầu có dịch hay không" so với mô phỏng list, qua nhiều lượt
def check_rule(n):
    arr = list(range(1, n + 1))
    first, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if left or cnt % 2 == 1:
            first += step
        step *= 2
        cnt //= 2
        arr = arr[1::2] if left else arr[::-1][1::2][::-1]
        left = not left
        assert arr == [first + k * step for k in range(cnt)], (n, arr, first, step, cnt)
    return first == arr[0]


print(all(check_rule(n) for n in range(1, 400)))      # True

True


## 7. Các cách tiếp theo

Tổng cộng **4 cách**: (1) mô phỏng list, (2) theo dõi đầu cấp số cộng, (3) đệ quy nhờ đối xứng gương, (4) theo dõi hai đầu `a1` và `an`.

### Cách 2 — Theo dõi phần tử đầu (đầu, bước, số lượng, hướng)

Giải thích từng đoạn:

- `first, step, cnt = 1, 1, n` và `left = True`.
- Trong khi `cnt > 1`: nếu `left` **hoặc** `cnt` lẻ thì `first += step`.
- Rồi `step <<= 1`, `cnt >>= 1`, đổi hướng `left = not left`.
- Trả về `first`.

In [11]:
def last_remaining_head(n):
    first, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if left or cnt % 2 == 1:
            first += step
        step <<= 1
        cnt >>= 1
        left = not left
    return first


for n, expected in examples:
    print(n, last_remaining_head(n), expected)
print([last_remaining_head(n) for n in range(1, 21)])

9 6 6
1 1 1
[1, 2, 2, 2, 2, 4, 4, 6, 6, 8, 8, 6, 6, 8, 8, 6, 6, 8, 8, 6]


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionHead:
    def lastRemaining(self, n: int) -> int:
        first, step, cnt, left = 1, 1, n, True
        while cnt > 1:
            if left or cnt % 2 == 1:
                first += step
            step <<= 1
            cnt >>= 1
            left = not left
        return first


print([SolutionHead().lastRemaining(n) for n, _ in examples], SolutionHead().lastRemaining(10**9))

[6, 1] 534765398


Độ phức tạp: số vòng lặp `⌊log2 n⌋` ⇒ **Time `O(log n)`**, **Space `O(1)`**.

### Cách 3 — Đệ quy nhờ đối xứng gương

Gọi `f(n)` là đáp án khi bắt đầu **từ trái** trên `1..n`. Sau lượt trái ta còn `2, 4, ..., 2m` với `m = n // 2`. Phần tử `2i` ứng với số `i` trong danh sách `1..m`. Lượt kế tiếp đi **từ phải** trên danh sách đó. Theo đối xứng gương (mục 3.6), đáp án "bắt đầu từ phải trên `1..m`" là `m + 1 - f(m)`. Vậy:

`f(1) = 1`,  `f(n) = 2 · (m + 1 − f(m))`  với `m = n // 2`.

Kiểm tra `n = 9`: `m = 4`; `f(4) = 2·(2+1−f(2))`, `f(2) = 2·(1+1−f(1)) = 2` ⇒ `f(4) = 2·1 = 2`; `f(9) = 2·(4+1−2) = 6` ✓.

In [13]:
def last_remaining_rec(n):
    if n == 1:
        return 1
    m = n // 2
    return 2 * (m + 1 - last_remaining_rec(m))


for n, expected in examples:
    print(n, last_remaining_rec(n), expected)
print([last_remaining_rec(n) for n in range(1, 21)])

9 6 6
1 1 1
[1, 2, 2, 2, 2, 4, 4, 6, 6, 8, 8, 6, 6, 8, 8, 6, 6, 8, 8, 6]


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionRec:
    def lastRemaining(self, n: int) -> int:
        if n == 1:
            return 1
        return 2 * (n // 2 + 1 - self.lastRemaining(n // 2))


print([SolutionRec().lastRemaining(n) for n, _ in examples], SolutionRec().lastRemaining(10**9))

[6, 1] 534765398


Độ phức tạp: độ sâu đệ quy `⌊log2 n⌋ ≈ 30` ⇒ **Time `O(log n)`**, **Space `O(log n)`** (ngăn xếp gọi hàm). Có thể "trải" đệ quy thành vòng lặp: ghi lại các giá trị `n, n//2, n//4, ..., 1` rồi tính ngược từ `f(1) = 1` lên — cùng độ phức tạp.

### Cách 4 — Theo dõi cả hai đầu (`a1` và `an`)

Thay vì lý luận "đầu có bị xóa không", ta theo dõi **cả hai đầu** của cấp số cộng.

- Lượt **trái → phải**: đầu luôn bị xóa ⇒ `a1 += step`. Đuôi bị xóa khi `cnt` lẻ (vì số cuối nằm ở vị trí lẻ từ trái) ⇒ nếu `cnt` lẻ thì `an -= step`.
- Lượt **phải → trái**: đuôi luôn bị xóa ⇒ `an -= step`. Đầu bị xóa khi `cnt` lẻ ⇒ nếu `cnt` lẻ thì `a1 += step`.
- Sau mỗi lượt: `cnt //= 2`, `step *= 2`. Khi `cnt == 1`, `a1 == an` là đáp án.

Cách này đối xứng, dễ kiểm tra (`a1 + (cnt-1)*step == an` ở mọi thời điểm).

In [15]:
def last_remaining_ends(n):
    a1, an, step, cnt, i = 1, n, 1, n, 0
    while cnt > 1:
        if i % 2 == 0:               # lượt trái -> phải
            a1 += step
            if cnt % 2:
                an -= step
        else:                        # lượt phải -> trái
            an -= step
            if cnt % 2:
                a1 += step
        cnt >>= 1
        step <<= 1
        i += 1
        assert a1 + (cnt - 1) * step == an     # bất biến: hai đầu luôn nhất quán
    return a1


for n, expected in examples:
    print(n, last_remaining_ends(n), expected)
print([last_remaining_ends(n) for n in range(1, 21)])

9 6 6
1 1 1
[1, 2, 2, 2, 2, 4, 4, 6, 6, 8, 8, 6, 6, 8, 8, 6, 6, 8, 8, 6]


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionEnds:
    def lastRemaining(self, n: int) -> int:
        a1, an, step, cnt, i = 1, n, 1, n, 0
        while cnt > 1:
            if i % 2 == 0:
                a1 += step
                if cnt % 2:
                    an -= step
            else:
                an -= step
                if cnt % 2:
                    a1 += step
            cnt >>= 1
            step <<= 1
            i += 1
        return a1


print([SolutionEnds().lastRemaining(n) for n, _ in examples], SolutionEnds().lastRemaining(10**9))

[6, 1] 534765398


Độ phức tạp: **Time `O(log n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

### Bảng trạng thái cho Cách 2 với `n = 9`

| Lượt `i` | Hướng | `cnt` | `cnt` lẻ? | `first` trước | Có dịch đầu? | `first` sau | `step` sau | `cnt` sau |
|---|---|---|---|---|---|---|---|---|
| 0 | trái→phải | 9 | có | 1 | **có** (luôn khi đi trái) | 1 + 1 = 2 | 2 | 4 |
| 1 | phải→trái | 4 | không | 2 | không (cnt chẵn) | 2 | 4 | 2 |
| 2 | trái→phải | 2 | không | 2 | **có** | 2 + 4 = 6 | 8 | 1 |

`cnt = 1` ⇒ dừng, đáp án `6` ✓. Bên cạnh, danh sách thật sau mỗi lượt là `[2,4,6,8]`, `[2,6]`, `[6]`, khớp `first + k·step` với `step = 2, 4, 8`.

In [17]:
def trace_head(n):
    first, step, cnt, left = 1, 1, n, True
    arr = list(range(1, n + 1))                         # chỉ để đối chiếu khi n nhỏ
    i = 0
    print(f"bắt đầu: first={first} step={step} cnt={cnt}  danh sách={arr}")
    while cnt > 1:
        direction = "trái->phải" if left else "phải->trái"
        shift = left or cnt % 2 == 1
        if shift:
            first += step
        step <<= 1
        cnt >>= 1
        arr = arr[1::2] if left else arr[::-1][1::2][::-1]
        print(f"lượt {i} {direction}: dịch đầu? {str(shift):5} -> first={first} step={step} cnt={cnt}  danh sách thật={arr}")
        assert arr == [first + k * step for k in range(cnt)]
        left = not left
        i += 1
    return first


print("kết quả:", trace_head(9))
print()
print("kết quả:", trace_head(12))

bắt đầu: first=1 step=1 cnt=9  danh sách=[1, 2, 3, 4, 5, 6, 7, 8, 9]
lượt 0 trái->phải: dịch đầu? True  -> first=2 step=2 cnt=4  danh sách thật=[2, 4, 6, 8]
lượt 1 phải->trái: dịch đầu? False -> first=2 step=4 cnt=2  danh sách thật=[2, 6]
lượt 2 trái->phải: dịch đầu? True  -> first=6 step=8 cnt=1  danh sách thật=[6]
kết quả: 6

bắt đầu: first=1 step=1 cnt=12  danh sách=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
lượt 0 trái->phải: dịch đầu? True  -> first=2 step=2 cnt=6  danh sách thật=[2, 4, 6, 8, 10, 12]
lượt 1 phải->trái: dịch đầu? False -> first=2 step=4 cnt=3  danh sách thật=[2, 6, 10]
lượt 2 trái->phải: dịch đầu? True  -> first=6 step=8 cnt=1  danh sách thật=[6]
kết quả: 6


### Trace cho Cách 3 (đệ quy) với `n = 9`

Gọi xuống: `f(9) → f(4) → f(2) → f(1)`. Rồi trả kết quả ngược lên: `f(1) = 1`, `f(2) = 2·(1+1−1) = 2`, `f(4) = 2·(2+1−2) = 2`, `f(9) = 2·(4+1−2) = 6`.

In [18]:
def trace_rec(n, depth=0):
    pad = "  " * depth
    if n == 1:
        print(f"{pad}f(1) = 1   (base case)")
        return 1
    m = n // 2
    print(f"{pad}f({n}): m = {m}, cần f({m})")
    sub = trace_rec(m, depth + 1)
    res = 2 * (m + 1 - sub)
    print(f"{pad}f({n}) = 2*({m}+1-{sub}) = {res}")
    return res


print("kết quả:", trace_rec(9))

f(9): m = 4, cần f(4)
  f(4): m = 2, cần f(2)
    f(2): m = 1, cần f(1)
      f(1) = 1   (base case)
    f(2) = 2*(1+1-1) = 2
  f(4) = 2*(2+1-2) = 2
f(9) = 2*(4+1-2) = 6
kết quả: 6


### Trace cho Cách 4 (hai đầu) với `n = 9`

| Lượt | Hướng | `cnt` | `a1` | `an` | `step` | `cnt` sau |
|---|---|---|---|---|---|---|
| 0 | trái→phải | 9 (lẻ) | 1+1 = 2 | 9−1 = 8 (vì lẻ) | 2 | 4 |
| 1 | phải→trái | 4 (chẵn) | 2 (chẵn nên không dịch) | 8−2 = 6 | 4 | 2 |
| 2 | trái→phải | 2 (chẵn) | 2+4 = 6 | 6 (chẵn nên không đổi) | 8 | 1 |

In [19]:
def trace_ends(n):
    a1, an, step, cnt, i = 1, n, 1, n, 0
    print(f"bắt đầu: a1={a1} an={an} step={step} cnt={cnt}")
    while cnt > 1:
        left = i % 2 == 0
        if left:
            a1 += step
            if cnt % 2:
                an -= step
        else:
            an -= step
            if cnt % 2:
                a1 += step
        cnt >>= 1
        step <<= 1
        i += 1
        print(f"lượt {i - 1} {'trái->phải' if left else 'phải->trái'}: a1={a1} an={an} step={step} cnt={cnt}")
    return a1


print("kết quả:", trace_ends(9))

bắt đầu: a1=1 an=9 step=1 cnt=9
lượt 0 trái->phải: a1=2 an=8 step=2 cnt=4
lượt 1 phải->trái: a1=2 an=6 step=4 cnt=2
lượt 2 trái->phải: a1=6 an=6 step=8 cnt=1
kết quả: 6


## 9. Code nộp LeetCode

Chọn **Cách 2** (theo dõi đầu) làm bản nộp chính — ngắn, `O(log n)` thời gian, `O(1)` bộ nhớ:

```python
class Solution:
    def lastRemaining(self, n: int) -> int:
        first, step, cnt, left = 1, 1, n, True
        while cnt > 1:
            if left or cnt % 2 == 1:
                first += step
            step <<= 1
            cnt >>= 1
            left = not left
        return first
```

In [20]:
class Solution:
    def lastRemaining(self, n: int) -> int:
        first, step, cnt, left = 1, 1, n, True
        while cnt > 1:
            if left or cnt % 2 == 1:
                first += step
            step <<= 1
            cnt >>= 1
            left = not left
        return first


for n, expected in examples:
    got = Solution().lastRemaining(n)
    print(n, got, expected, got == expected)

9 6 6 True
1 1 1 True


## 10. Vì sao đúng

**Bất biến:** trước mỗi lượt, danh sách hiện tại gồm đúng các số `first, first+step, ..., first+(cnt−1)·step`, và đáp án cuối cùng là một phần tử của danh sách này.

*Khởi tạo:* `first = 1, step = 1, cnt = n` đúng.

*Duy trì:* xét một lượt. Gọi các phần tử `p_0, p_1, ..., p_{cnt−1}`.
- **Trái → phải:** xóa `p_0, p_2, p_4, ...`; còn `p_1, p_3, ...` — cấp số cộng đầu `p_1 = first + step`, bước `2·step`, số lượng `⌊cnt/2⌋`.
- **Phải → trái:** xóa `p_{cnt−1}, p_{cnt−3}, ...`. Phần tử còn lại là `p_{cnt−2}, p_{cnt−4}, ...`, tức các chỉ số cùng tính chẵn lẻ với `cnt` (cnt−2 ≡ cnt mod 2). Chỉ số nhỏ nhất của chúng: nếu `cnt` chẵn thì chỉ số `0` còn lại (`p_0`, đầu không đổi); nếu `cnt` lẻ thì chỉ số nhỏ nhất là `1` (đầu mới là `p_1 = first + step`). Bước vẫn là `2·step` và số lượng `⌊cnt/2⌋`.

Vậy quy tắc "lượt trái luôn dịch đầu; lượt phải dịch đầu khi `cnt` lẻ" đúng, `step *= 2`, `cnt //= 2` đúng.

*Kết thúc:* mỗi lượt `cnt` giảm còn `⌊cnt/2⌋`, tới một lúc `cnt = 1`; lúc đó danh sách chỉ còn `first` ⇒ đáp án. Số lượt là `⌊log2 n⌋`.

(Cách 3 đúng nhờ cùng nhận xét sau lượt trái: các số còn lại là `2·1, 2·2, ..., 2·m` và đối xứng gương: đáp án đi từ phải trên `1..m` là `m + 1 − f(m)`.)

## 11. Time/Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng list | `O(n)` | `O(n)` |
| 2. Đầu cấp số cộng | **`O(log n)`** | **`O(1)`** |
| 3. Đệ quy gương | `O(log n)` | `O(log n)` |
| 4. Hai đầu `a1`, `an` | `O(log n)` | `O(1)` |

Với `n = 10^9`, `log2 n ≈ 30` vòng lặp. Không có phép toán nào phụ thuộc `n` ngoài số vòng.

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

In [21]:
methods = [last_remaining_brute, last_remaining_head, last_remaining_rec, last_remaining_ends,
           SolutionHead().lastRemaining, SolutionRec().lastRemaining, SolutionEnds().lastRemaining,
           Solution().lastRemaining]

fixed = {1: 1, 2: 2, 3: 2, 4: 2, 5: 2, 6: 4, 7: 4, 8: 6, 9: 6, 10: 8, 12: 6, 16: 6}
for n, expected in fixed.items():
    for f in methods:
        assert f(n) == expected, (f, n, expected, f(n))
print("OK ví dụ & edge nhỏ:", fixed)

OK ví dụ & edge nhỏ: {1: 1, 2: 2, 3: 2, 4: 2, 5: 2, 6: 4, 7: 4, 8: 6, 9: 6, 10: 8, 12: 6, 16: 6}


### 12.2. Đối chiếu mọi `n` từ 1 đến 3000 với mô phỏng

In [22]:
for n in range(1, 3001):
    expected = last_remaining_brute(n)
    assert last_remaining_head(n) == expected, n
    assert last_remaining_rec(n) == expected, n
    assert last_remaining_ends(n) == expected, n
    assert Solution().lastRemaining(n) == expected, n
print("OK: 3000 giá trị, 4 cách khớp mô phỏng")

OK: 3000 giá trị, 4 cách khớp mô phỏng


### 12.3. Ngẫu nhiên có seed (đến 200000) và test giá trị rất lớn

In [23]:
import random
import time

rng = random.Random(390)
for _ in range(60):
    n = rng.randint(3000, 200000)
    assert Solution().lastRemaining(n) == last_remaining_brute(n), n
print("OK: 60 giá trị ngẫu nhiên đến 200000 khớp mô phỏng")

# n rất lớn: không dùng mô phỏng, đối chiếu 3 cách O(log n) với nhau
for n in [10**9, 10**9 - 1, 2**29, 2**29 + 1, 999999937, 536870911, 123456789]:
    results = {last_remaining_head(n), last_remaining_rec(n), last_remaining_ends(n)}
    assert len(results) == 1, (n, results)
    assert 1 <= results.copy().pop() <= n
    print(n, "->", results.pop())

t0 = time.perf_counter()
for _ in range(10000):
    Solution().lastRemaining(10**9)
print("10000 lần n=1e9:", round(time.perf_counter() - t0, 3), "giây")

OK: 60 giá trị ngẫu nhiên đến 200000 khớp mô phỏng
1000000000 -> 534765398
999999999 -> 534765056
536870912 -> 357913942
536870913 -> 357913942
999999937 -> 534765014
536870911 -> 268435456
123456789 -> 56614230
10000 lần n=1e9: 0.021 giây


### 12.4. Lũy thừa của 2 và quy luật

Quan sát `f(2^k)` để thấy quy luật (hiển thị, đối chiếu mô phỏng khi `2^k` nhỏ):

In [24]:
vals = []
for k in range(0, 21):
    n = 2 ** k
    v = Solution().lastRemaining(n)
    if n <= 2 ** 16:
        assert v == last_remaining_brute(n)
    vals.append(v)
print(vals)
# hai giá trị liên tiếp bằng nhau: f(2^(2j-1)) == f(2^(2j))? kiểm tra
print(all(vals[2 * j - 1] == vals[2 * j] for j in range(1, 10)))

[1, 2, 2, 6, 6, 22, 22, 86, 86, 342, 342, 1366, 1366, 5462, 5462, 21846, 21846, 87382, 87382, 349526, 349526]
True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: luôn dịch phần tử đầu ở mọi lượt (bỏ qua chẵn/lẻ khi đi từ phải)

In [25]:
def bug_always_shift(n):
    first, step, cnt = 1, 1, n
    while cnt > 1:
        first += step                       # SAI: lượt phải chỉ dịch khi cnt lẻ
        step <<= 1
        cnt >>= 1
    return first


print([bug_always_shift(n) for n in range(1, 13)])
print([Solution().lastRemaining(n) for n in range(1, 13)])
print("n=9:", bug_always_shift(9), "(đúng là 6)")

[1, 2, 2, 4, 4, 4, 4, 8, 8, 8, 8, 8]
[1, 2, 2, 2, 2, 4, 4, 6, 6, 8, 8, 6]
n=9: 8 (đúng là 6)


**Vì sao sai:** lượt phải → trái chỉ xóa phần tử đầu khi `cnt` lẻ. Với `cnt` chẵn (ví dụ `[2,4,6,8]` → `[2,6]`) phần tử đầu vẫn còn. **Sửa:** `if left or cnt % 2 == 1: first += step`.

### Lỗi 2: cập nhật `cnt` sai (làm tròn lên thay vì xuống)

In [26]:
def bug_cnt_round_up(n):
    first, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if left or cnt % 2 == 1:
            first += step
        step <<= 1
        cnt = (cnt + 1) // 2               # SAI: phải là cnt // 2
        left = not left
    return first


print([bug_cnt_round_up(n) for n in range(1, 13)])
print([Solution().lastRemaining(n) for n in range(1, 13)])

[1, 2, 2, 2, 8, 8, 6, 6, 8, 8, 6, 6]
[1, 2, 2, 2, 2, 4, 4, 6, 6, 8, 8, 6]


**Vì sao sai:** xóa các chỉ số `0, 2, 4, ...` (trái) từ `cnt` phần tử thì còn lại `cnt // 2`, không phải `(cnt + 1) // 2`. Giữ nhầm một phần tử sẽ làm vòng lặp chạy thêm lượt (với `cnt = 3`: ra `2` rồi `1` — thay vì `1` ngay) và lệch đáp án. **Sửa:** `cnt >>= 1` hoặc `cnt //= 2`.

### Lỗi 3: công thức đệ quy sai gương

In [27]:
def bug_rec_mirror(n):
    if n == 1:
        return 1
    return 2 * (n // 2 - bug_rec_mirror(n // 2))        # SAI: thiếu "+ 1" của phép gương m + 1 - x


def bug_rec_no_mirror(n):
    if n == 1:
        return 1
    return 2 * bug_rec_no_mirror(n // 2)                # SAI: quên đổi hướng


print([bug_rec_no_mirror(n) for n in range(2, 9)])
print([Solution().lastRemaining(n) for n in range(2, 9)])
try:
    print([bug_rec_mirror(n) for n in range(2, 9)])
except Exception as e:
    print(type(e).__name__, e)

[2, 2, 4, 4, 4, 4, 8]
[2, 2, 2, 2, 4, 4, 6]
[0, 0, 4, 4, 6, 6, 0]


**Vì sao sai:** phép gương trên danh sách `1..m` là `x → m + 1 − x`. Nếu viết `m − x` (thiếu `+1`) thì với `m = 1` ta được `0` ⇒ kết quả `0` (không thuộc `1..n`). Nếu bỏ hẳn gương, ta coi như lượt thứ hai cũng đi từ trái. **Sửa:** `2 * (n // 2 + 1 - f(n // 2))`.

### Lỗi 4: mô phỏng bằng list với `n` lớn (Time Limit / Memory Limit)

Mô phỏng tuyến tính theo `n`. Đo thời gian khi `n` tăng gấp đôi để thấy xu hướng:

In [28]:
import time

for n in [10**5, 2 * 10**5, 4 * 10**5, 8 * 10**5]:
    t0 = time.perf_counter()
    last_remaining_brute(n)
    print(f"n = {n:7}: {time.perf_counter() - t0:.4f}s")
print("Với n = 10**9 (gấp ~1250 lần n=8e5) cần hàng chục GB RAM => MLE/TLE")

n =  100000: 0.0041s
n =  200000: 0.0089s
n =  400000: 0.0200s
n =  800000: 0.0403s
Với n = 10**9 (gấp ~1250 lần n=8e5) cần hàng chục GB RAM => MLE/TLE


**Vì sao sai:** thời gian và bộ nhớ tỉ lệ thuận `n`; với `n = 10^9` thì không thể tạo nổi list. **Sửa:** chỉ theo dõi `(first, step, cnt)` như Cách 2.

## 14. Mẹo & nhận dạng pattern

- Thấy **`n` lên tới `10^9`** và quy trình "xóa một nửa mỗi bước" ⇒ nghĩ ngay **`O(log n)`**: theo dõi vài tham số của dãy chứ không liệt kê.
- Mỗi lượt giảm một nửa số phần tử ⇒ độ sâu `log2 n`; dãy còn lại thường là **cấp số cộng** ⇒ ba số `(đầu, bước, đếm)` là đủ.
- Bài **luân phiên hướng** ⇒ dùng **đối xứng gương** để quy về "luôn đi từ trái" (hoặc một cờ boolean `left`).
- Hãy **viết mô phỏng brute** trước, in ra `n = 1..20`, tìm quy luật, rồi dùng chính mô phỏng làm *oracle* kiểm chứng công thức (đúng như notebook này).
- Họ hàng: **Josephus problem** (xóa phần tử thứ `k`), **Bit manipulation** khi bước nhân đôi, *Find the winner of the circular game*.
- Cẩn thận chuyện **chẵn/lẻ** — hầu hết lỗi trong bài này là sai một đơn vị ở tính chẵn/lẻ của `cnt`.

## 15. Biến thể

1. **Bắt đầu từ phải trước:** gọi `g(n)`. Theo đối xứng gương `g(n) = n + 1 − f(n)`.
2. **Số lượt xóa** cho tới khi còn một số: `⌊log2 n⌋ = n.bit_length() - 1`.
3. **Liệt kê thứ tự các số bị xóa** (ghi lại phần bị xóa trong mô phỏng ở Cách 1): hữu ích để kiểm tra bằng mắt cho `n` nhỏ.
4. **Vòng tròn (Josephus)**: xóa mỗi người thứ 2 theo vòng; có công thức `2·(n − 2^k) + 1` với `2^k` là lũy thừa 2 lớn nhất `≤ n`.

Kiểm chứng biến thể 1 và 2 bằng mô phỏng:

In [29]:
def last_remaining_right_first(n):
    return n + 1 - Solution().lastRemaining(n)


def brute_right_first(n):
    arr = list(range(1, n + 1))
    from_left = False
    while len(arr) > 1:
        arr = arr[1::2] if from_left else arr[::-1][1::2][::-1]
        from_left = not from_left
    return arr[0]


assert all(last_remaining_right_first(n) == brute_right_first(n) for n in range(1, 1500))


def rounds(n):
    r = 0
    while n > 1:
        n >>= 1
        r += 1
    return r


assert all(rounds(n) == n.bit_length() - 1 for n in range(1, 5000))
print("OK biến thể: bắt đầu từ phải, và số lượt = bit_length - 1")
print([last_remaining_right_first(n) for n in range(1, 13)])

OK biến thể: bắt đầu từ phải, và số lượt = bit_length - 1
[1, 1, 2, 3, 4, 3, 4, 3, 4, 3, 4, 7]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng list | Cắt lát `[1::2]`, đảo hướng | `O(n)` | `O(n)` | Hiểu đề, **làm oracle** cho `n` nhỏ |
| 2. Đầu cấp số cộng | `first/step/cnt`; đầu dịch nếu trái hoặc `cnt` lẻ | `O(log n)` | `O(1)` | **Cách nộp chuẩn** |
| 3. Đệ quy gương | `f(n) = 2(m+1−f(m))` | `O(log n)` | `O(log n)` | Hiểu cấu trúc đệ quy, code ngắn nhất |
| 4. Hai đầu | Theo dõi `a1` và `an` đối xứng | `O(log n)` | `O(1)` | Cần kiểm tra nhất quán hai đầu |

## 17. Bài tập vận dụng (dễ → khó)

### Bài 1 (đoán output)

Đoán `result_1` trước khi chạy hàm.

In [30]:
def sample_slices():
    a = [10, 20, 30, 40, 50, 60]
    return a[1::2], a[::-1][1::2][::-1]


# Đoán: sample_slices() trả về cặp (list1, list2) nào?
answer_1 = None   # TODO: điền một tuple gồm hai list, ví dụ ([..], [..])

### Bài 2 (tính tay)

Tính tay `lastRemaining(n)` cho `n = 12` theo từng lượt (ghi lại danh sách sau mỗi lượt) rồi điền đáp án. Sau đó kiểm tra bằng cách đã học ở mục 4.

In [31]:
answer_2 = None   # TODO: điền một số nguyên (làm tay trước, đừng gọi hàm)

### Bài 3 (điền TODO)

Hoàn thiện Cách 2 (theo dõi đầu cấp số cộng).

In [32]:
def last_remaining_todo(n):
    first, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        # TODO 1: khi nào first += step ?  (gợi ý: left hoặc cnt lẻ)
        ...
        # TODO 2: cập nhật step, cnt và hướng
        ...
    return first

### Bài 4 (tự viết lại)

Viết hàm đệ quy `last_remaining_rec2(n)` theo công thức `f(1) = 1`, `f(n) = 2·(n//2 + 1 − f(n//2))`. Hãy tự kiểm chứng với mô phỏng ở các `n` nhỏ.

In [33]:
def last_remaining_rec2(n):
    # TODO: tự viết
    pass

### Bài 5 (biến thể / mini challenge)

Viết `last_remaining_right_first2(n)`: giống đề nhưng **lượt đầu từ phải sang trái** (rồi trái, rồi phải...). Thử không dùng mô phỏng list; gợi ý dùng đối xứng gương hoặc đổi giá trị khởi tạo của cờ `left`. (Với `n = 9` đáp án là `4`.)

In [34]:
def last_remaining_right_first2(n):
    # TODO: tự viết
    pass

## 18. Gợi ý (3 mức)

**Bài 1**
- Mức 1: `a[1::2]` giữ các chỉ số lẻ.
- Mức 2: `a[::-1]` đảo thành `[60, 50, 40, 30, 20, 10]`, rồi `[1::2]` giữ chỉ số lẻ của nó.
- Mức 3: kết quả có dạng `([20, 40, 60], [10, 30, 50])`. Kiểm tra bằng cách chạy.

**Bài 2**
- Mức 1: `[1..12]` lượt trái xóa các số lẻ.
- Mức 2: còn `[2,4,6,8,10,12]`; lượt phải xóa `12, 8, 4`.
- Mức 3: còn `[2,6,10]`; lượt trái xóa `2` và `10`.

**Bài 3**
- Mức 1: `if left or cnt % 2 == 1:` rồi `first += step`.
- Mức 2: `step <<= 1`, `cnt >>= 1`.
- Mức 3: đừng quên `left = not left` ở cuối mỗi vòng.

**Bài 4**
- Mức 1: base case `n == 1` trả `1`.
- Mức 2: `m = n // 2`.
- Mức 3: `return 2 * (m + 1 - last_remaining_rec2(m))`.

**Bài 5**
- Mức 1: ghi nhớ `g(n) = n + 1 - f(n)` (đối xứng gương toàn bộ).
- Mức 2: hoặc dùng Cách 2 với `left = False` ban đầu *(nhưng quy tắc dịch đầu thay đổi theo hướng; hãy đối chiếu mô phỏng)*.
- Mức 3: cách chắc chắn nhất: `return n + 1 - Solution().lastRemaining(n)`.

## 19. Đáp án tham khảo

In [35]:
# Bài 1
answer_1 = ([20, 40, 60], [10, 30, 50])
assert sample_slices() == answer_1

# Bài 2: [1..12] -> [2,4,6,8,10,12] -> [2,6,10] -> [6]
answer_2 = 6
assert answer_2 == Solution().lastRemaining(12) == last_remaining_brute(12)


# Bài 3
def last_remaining_todo(n):
    first, step, cnt, left = 1, 1, n, True
    while cnt > 1:
        if left or cnt % 2 == 1:
            first += step
        step <<= 1
        cnt >>= 1
        left = not left
    return first


assert all(last_remaining_todo(n) == last_remaining_brute(n) for n in range(1, 500))


# Bài 4
def last_remaining_rec2(n):
    if n == 1:
        return 1
    m = n // 2
    return 2 * (m + 1 - last_remaining_rec2(m))


assert all(last_remaining_rec2(n) == last_remaining_brute(n) for n in range(1, 500))


# Bài 5
def last_remaining_right_first2(n):
    return n + 1 - Solution().lastRemaining(n)


assert last_remaining_right_first2(9) == 4
assert all(last_remaining_right_first2(n) == brute_right_first(n) for n in range(1, 500))
print("Tất cả đáp án đã chạy đúng")

Tất cả đáp án đã chạy đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
first, step, cnt, left = 1, 1, n, True
while cnt > 1:
    if left or cnt % 2 == 1:   # trái: luôn dịch đầu; phải: chỉ dịch khi cnt lẻ
        first += step
    step <<= 1                 # bước nhân đôi
    cnt >>= 1                  # số lượng giảm nửa (làm tròn xuống)
    left = not left            # đổi hướng
return first
```

Công thức đệ quy tương đương: `f(1) = 1`, `f(n) = 2 * (n // 2 + 1 - f(n // 2))`.

### Checklist

- [ ] Không tạo list `1..n` (n tới `10^9`).
- [ ] `first, step, cnt` khởi tạo `1, 1, n`; vòng lặp `while cnt > 1`.
- [ ] Điều kiện dịch đầu: `left or cnt % 2 == 1`.
- [ ] `cnt //= 2` (làm tròn xuống), `step *= 2`, đổi hướng cuối mỗi vòng.
- [ ] Test: `n = 1 → 1`, `n = 2 → 2`, `n = 9 → 6`, `n = 10^9` (không lỗi, chạy tức thì).
- [ ] Đã đối chiếu với mô phỏng list trên mọi `n` nhỏ.